In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np

from google.colab import drive
drive.mount("/content/drive")

DATA_RAW = Path("/content/drive/MyDrive/STAT596/Project596_datafiles")
DATA_PROCESSED = DATA_RAW / "processed_data"
TABLES = DATA_RAW / "tables"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

sd_tracts_file = DATA_RAW / "sd_census_tracts_clean.geojson"
south_bay_tracts_file = DATA_RAW / "south_bay_census_tracts.geojson"
tract_attributes_file = DATA_RAW / "sd_census_tracts_clean_attributes.csv"
beach_stations_file = DATA_PROCESSED / "beach_monitoring_stations_clean.csv"

output_geojson = DATA_PROCESSED / "south_bay_census_tracts_proximity.geojson"
output_csv = DATA_PROCESSED / "south_bay_census_tracts_proximity_attributes.csv"
summary_file = TABLES / "census_spatial_layers_processing_summary.csv"

files_to_check = [
    sd_tracts_file,
    south_bay_tracts_file,
    tract_attributes_file,
    beach_stations_file
]

file_check = pd.DataFrame({
    "file_name": [file.name for file in files_to_check],
    "path": [str(file) for file in files_to_check],
    "exists": [file.exists() for file in files_to_check]
})

print("Input check complete.")
display(file_check)

print("\nPlanned output files:")
print(output_geojson)
print(output_csv)
print(summary_file)

if not file_check["exists"].all():
    missing = file_check.loc[~file_check["exists"], "file_name"].tolist()
    raise FileNotFoundError(f"Missing files: {missing}")

Mounted at /content/drive
Input check complete.


,file_name,path,exists
0,sd_census_tracts_clean.geojson,/content/drive/MyDrive/STAT596/Project596_data...,True
1,south_bay_census_tracts.geojson,/content/drive/MyDrive/STAT596/Project596_data...,True
2,sd_census_tracts_clean_attributes.csv,/content/drive/MyDrive/STAT596/Project596_data...,True
3,beach_monitoring_stations_clean.csv,/content/drive/MyDrive/STAT596/Project596_data...,True



Planned output files:
/content/drive/MyDrive/STAT596/Project596_datafiles/processed_data/south_bay_census_tracts_proximity.geojson
/content/drive/MyDrive/STAT596/Project596_datafiles/processed_data/south_bay_census_tracts_proximity_attributes.csv
/content/drive/MyDrive/STAT596/Project596_datafiles/tables/census_spatial_layers_processing_summary.csv


In [ ]:
# Checks census tract layers and beach station metadata before cleaning.

sd_tracts = gpd.read_file(sd_tracts_file)
south_bay_tracts = gpd.read_file(south_bay_tracts_file)
tract_attributes = pd.read_csv(tract_attributes_file)
beach_stations = pd.read_csv(beach_stations_file)

gdf_check = pd.DataFrame({
    "layer": ["sd_tracts", "south_bay_tracts"],
    "rows": [len(sd_tracts), len(south_bay_tracts)],
    "columns": [sd_tracts.shape[1], south_bay_tracts.shape[1]],
    "crs": [str(sd_tracts.crs), str(south_bay_tracts.crs)],
    "geometry_types": [
        ", ".join(sd_tracts.geometry.geom_type.dropna().unique()),
        ", ".join(south_bay_tracts.geometry.geom_type.dropna().unique())
    ],
    "missing_geometry": [
        sd_tracts.geometry.isna().sum(),
        south_bay_tracts.geometry.isna().sum()
    ]
})

table_check = pd.DataFrame({
    "table": ["tract_attributes", "beach_stations"],
    "rows": [len(tract_attributes), len(beach_stations)],
    "columns": [tract_attributes.shape[1], beach_stations.shape[1]]
})

tract_id_candidates = ["GEOID", "geoid", "GEOID20", "TRACTCE", "TRACTCE20", "NAMELSAD", "NAME"]
station_key_columns = ["station_id", "beach_name_id", "station_name", "beach_name", "station_lat", "station_lon"]

available_tract_id_columns = [col for col in tract_id_candidates if col in south_bay_tracts.columns]
available_station_columns = [col for col in station_key_columns if col in beach_stations.columns]

print("Output verification complete.")

print("\nSpatial layer check:")
display(gdf_check)

print("\nTable check:")
display(table_check)

print("\nAvailable tract ID columns:")
print(available_tract_id_columns)

print("\nAvailable beach station key columns:")
print(available_station_columns)

print("\nSouth Bay tract columns:")
print(list(south_bay_tracts.columns))

print("\nBeach station first rows:")
display(beach_stations[available_station_columns].head())

Output verification complete.

Spatial layer check:


,layer,rows,columns,crs,geometry_types,missing_geometry
0,sd_tracts,737,14,EPSG:2230,Polygon,0
1,south_bay_tracts,263,14,EPSG:2230,Polygon,0



Table check:


,table,rows,columns
0,tract_attributes,737,13
1,beach_stations,284,34



Available tract ID columns:
['geoid']

Available beach station key columns:
['station_id', 'beach_name_id', 'station_name', 'beach_name', 'station_lat', 'station_lon']

South Bay tract columns:
['tract_id', 'tract_name', 'area_sq_km', 'centroid_lat', 'centroid_lon', 'south_bay_study_area', 'geoid', 'name', 'namelsad', 'aland', 'awater', 'intptlat', 'intptlon', 'geometry']

Beach station first rows:


,station_id,beach_name_id,station_name,beach_name,station_lat,station_lon
0,277,413,CRK Agua Hedionda,Agua Hedionda Lagoon,33.144612,-117.326613
1,385,413,EH-450,Agua Hedionda Lagoon,33.145000,-117.340900
2,386,413,EH-452,Agua Hedionda Lagoon,33.144618,-117.337290
3,472,413,LAG Agua Hedionda,Agua Hedionda Lagoon,33.139292,-117.329032
4,88,420,All_SanDiego_County_Beaches,All_SanDiego_County,33.386900,-117.596000


In [ ]:
# Standardizes tract IDs for the census layers.similar to lecture activities
def clean_geoid(series):
    return (
        series.astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
        .str.zfill(11)
    )

sd_tracts["geoid"] = clean_geoid(sd_tracts["geoid"])
south_bay_tracts["geoid"] = clean_geoid(south_bay_tracts["geoid"])
tract_attributes["geoid"] = clean_geoid(tract_attributes["geoid"])

geoid_check = pd.DataFrame({
    "dataset": ["sd_tracts", "south_bay_tracts", "tract_attributes"],
    "rows": [len(sd_tracts), len(south_bay_tracts), len(tract_attributes)],
    "unique_geoid": [
        sd_tracts["geoid"].nunique(),
        south_bay_tracts["geoid"].nunique(),
        tract_attributes["geoid"].nunique()
    ],
    "missing_geoid": [
        sd_tracts["geoid"].isna().sum(),
        south_bay_tracts["geoid"].isna().sum(),
        tract_attributes["geoid"].isna().sum()
    ],
    "duplicate_geoid": [
        sd_tracts["geoid"].duplicated().sum(),
        south_bay_tracts["geoid"].duplicated().sum(),
        tract_attributes["geoid"].duplicated().sum()
    ]
})

south_bay_in_attributes = south_bay_tracts["geoid"].isin(tract_attributes["geoid"]).sum()

print("Processing summary.")
display(geoid_check)

print("\nSouth Bay tract GEOIDs found in attribute table:")
print(f"{south_bay_in_attributes} of {len(south_bay_tracts)}")

print("\nSouth Bay tract preview:")
display(south_bay_tracts[["geoid", "tract_id", "tract_name", "area_sq_km"]].head())

Processing summary.


,dataset,rows,unique_geoid,missing_geoid,duplicate_geoid
0,sd_tracts,737,737,0,0
1,south_bay_tracts,263,263,0,0
2,tract_attributes,737,737,0,0



South Bay tract GEOIDs found in attribute table:
263 of 263

South Bay tract preview:


,geoid,tract_id,tract_name,area_sq_km
0,06073010015,06073010015,Census Tract 100.15,40.475321
1,06073021900,06073021900,Census Tract 219,9.502008
2,06073013421,06073013421,Census Tract 134.21,2.476405
3,06073000400,06073000400,Census Tract 4,1.177453
4,06073000600,06073000600,Census Tract 6,0.932841


In [ ]:
# preparing the projected tract layer and beach station point layer

target_crs = "EPSG:3310"

south_bay_tracts_proj = south_bay_tracts.to_crs(target_crs)

south_bay_tracts_proj["tract_point"] = south_bay_tracts_proj.geometry.representative_point()

beach_stations_clean = beach_stations.dropna(subset=["station_lat", "station_lon"]).copy()

beach_points = gpd.GeoDataFrame(
    beach_stations_clean,
    geometry=gpd.points_from_xy(
        beach_stations_clean["station_lon"],
        beach_stations_clean["station_lat"]
    ),
    crs="EPSG:4326"
)

beach_points_proj = beach_points.to_crs(target_crs)

spatial_prep_check = pd.DataFrame({
    "layer": ["south_bay_tracts_proj", "beach_points_proj"],
    "rows": [len(south_bay_tracts_proj), len(beach_points_proj)],
    "crs": [str(south_bay_tracts_proj.crs), str(beach_points_proj.crs)],
    "missing_geometry": [
        south_bay_tracts_proj.geometry.isna().sum(),
        beach_points_proj.geometry.isna().sum()
    ]
})

print("Processing summary.")
display(spatial_prep_check)

print("\nBeach station coordinate check:")
display(beach_points_proj[["station_id", "station_name", "beach_name", "station_lat", "station_lon"]].head())

Processing summary.


,layer,rows,crs,missing_geometry
0,south_bay_tracts_proj,263,EPSG:3310,0
1,beach_points_proj,284,EPSG:3310,0



Beach station coordinate check:


,station_id,station_name,beach_name,station_lat,station_lon
0,277,CRK Agua Hedionda,Agua Hedionda Lagoon,33.144612,-117.326613
1,385,EH-450,Agua Hedionda Lagoon,33.145000,-117.340900
2,386,EH-452,Agua Hedionda Lagoon,33.144618,-117.337290
3,472,LAG Agua Hedionda,Agua Hedionda Lagoon,33.139292,-117.329032
4,88,All_SanDiego_County_Beaches,All_SanDiego_County,33.386900,-117.596000


In [ ]:
#adding nearest beach station distance to each South Bay census tract
tract_points = south_bay_tracts_proj[["geoid", "tract_name", "tract_point"]].copy()
tract_points = gpd.GeoDataFrame(
    tract_points,
    geometry="tract_point",
    crs=south_bay_tracts_proj.crs
)

beach_points_small = beach_points_proj[[
    "station_id",
    "station_name",
    "beach_name",
    "geometry"
]].copy()

nearest_beach = gpd.sjoin_nearest(
    tract_points,
    beach_points_small,
    how="left",
    distance_col="nearest_beach_station_m"
)

nearest_beach = nearest_beach.rename(columns={
    "station_id": "nearest_beach_station_id",
    "station_name": "nearest_beach_station_name",
    "beach_name": "nearest_beach_name"
})

nearest_beach["nearest_beach_station_km"] = nearest_beach["nearest_beach_station_m"] / 1000

nearest_beach = nearest_beach[[
    "geoid",
    "nearest_beach_station_id",
    "nearest_beach_station_name",
    "nearest_beach_name",
    "nearest_beach_station_m",
    "nearest_beach_station_km"
]]

south_bay_proximity_proj = south_bay_tracts_proj.merge(
    nearest_beach,
    on="geoid",
    how="left"
)

check = pd.DataFrame({
    "rows": [len(south_bay_proximity_proj)],
    "missing_nearest_station": [south_bay_proximity_proj["nearest_beach_station_id"].isna().sum()],
    "min_distance_km": [south_bay_proximity_proj["nearest_beach_station_km"].min()],
    "max_distance_km": [south_bay_proximity_proj["nearest_beach_station_km"].max()]
})

print("Processing summary.")
display(check)

display(south_bay_proximity_proj[[
    "geoid",
    "tract_name",
    "nearest_beach_name",
    "nearest_beach_station_name",
    "nearest_beach_station_km"
]].head())

Processing summary.


,rows,missing_nearest_station,min_distance_km,max_distance_km
0,263,0,0.090363,31.599471


,geoid,tract_name,nearest_beach_name,nearest_beach_station_name,nearest_beach_station_km
0,06073010015,Census Tract 100.15,Tijana River,TJ-010,9.408292
1,06073021900,Census Tract 219,San Diego Bay,EH-140,1.004635
2,06073013421,Census Tract 134.21,San Diego Bay,EH-566,6.921795
3,06073000400,Census Tract 4,San Diego Bay,EH-535,3.994969
4,06073000600,Census Tract 6,San Diego Bay,EH-560,4.351398


In [ ]:
# looking for beach grouping fields that can support South Bay proximity

group_words = ["group", "study", "area", "region", "impact", "category"]

possible_group_cols = [
    col for col in beach_stations.columns
    if any(word in col.lower() for word in group_words)
]

south_bay_words = [
    "tijuana",
    "tijana",
    "imperial",
    "border",
    "coronado",
    "silver strand",
    "south bay"
]

beach_name_matches = beach_stations[
    beach_stations["beach_name"].str.lower().str.contains(
        "|".join(south_bay_words),
        na=False
    )
].copy()

station_name_matches = beach_stations[
    beach_stations["station_name"].str.lower().str.contains(
        "|".join(south_bay_words),
        na=False
    )
].copy()

print("Attribute check complete.")

print("\nPossible grouping columns:")
print(possible_group_cols)

print("\nBeach name matches:")
display(beach_name_matches[[
    "station_id",
    "station_name",
    "beach_name",
    "station_lat",
    "station_lon"
]].drop_duplicates().sort_values(["beach_name", "station_name"]))

print("\nStation name matches:")
display(station_name_matches[[
    "station_id",
    "station_name",
    "beach_name",
    "station_lat",
    "station_lon"
]].drop_duplicates().sort_values(["beach_name", "station_name"]))

Attribute check complete.

Possible grouping columns:
['regional_board', 'regional_board_name']

Beach name matches:


,station_id,station_name,beach_name,station_lat,station_lon
8,455,IB-010,Border Field State Park,32.535950,-117.124300
9,456,IB-020,Border Field State Park,32.543410,-117.125228
28,1152,EH-575,Coronado Cays (NR),32.628128,-117.136215
29,1153,EH-576,Coronado Cays (NR),32.630989,-117.135924
30,325,EH-050,Coronado City beaches,32.681840,-117.185260
31,326,EH-053,Coronado City beaches,32.682447,-117.188144
32,327,EH-056,Coronado City beaches,32.683654,-117.191172
33,1038,IB-079,Coronado City beaches,32.674066,-117.172300
34,462,IB-080,Coronado City beaches,32.677500,-117.177270
35,328,EH-060,Coronado north beach,32.685586,-117.194319



Station name matches:


,station_id,station_name,beach_name,station_lat,station_lon
248,651,RIV Tijuana,Tijana River,32.54763,-117.05403


In [ ]:
# tijuana River / Imperial Beach impact-area stations based on names.

impact_words = [
    "tijuana",
    "tijana",
    "border field",
    "imperial beach",
    "tijuana slough"
]

impact_pattern = "|".join(impact_words)

beach_points_proj["impact_area_station"] = (
    beach_points_proj["beach_name"].str.lower().str.contains(impact_pattern, na=False)
    | beach_points_proj["station_name"].str.lower().str.contains(impact_pattern, na=False)
)

impact_beach_points_proj = beach_points_proj[beach_points_proj["impact_area_station"]].copy()

impact_check = pd.DataFrame({
    "all_beach_stations": [len(beach_points_proj)],
    "impact_area_stations": [len(impact_beach_points_proj)],
    "missing_lat": [impact_beach_points_proj["station_lat"].isna().sum()],
    "missing_lon": [impact_beach_points_proj["station_lon"].isna().sum()]
})

print("Processing summary.")
display(impact_check)

display(impact_beach_points_proj[[
    "station_id",
    "station_name",
    "beach_name",
    "station_lat",
    "station_lon"
]].sort_values(["beach_name", "station_name"]))

Processing summary.


,all_beach_stations,impact_area_stations,missing_lat,missing_lon
0,284,18,0,0


,station_id,station_name,beach_name,station_lat,station_lon
8,455,IB-010,Border Field State Park,32.535950,-117.124300
9,456,IB-020,Border Field State Park,32.543410,-117.125228
45,319,EH-010,"Imperial Beach municipal beach, other",32.572665,-117.133201
46,320,EH-020,"Imperial Beach municipal beach, other",32.576873,-117.132802
47,1113,IB-045,"Imperial Beach municipal beach, other",32.569389,-117.133023
48,459,IB-050,"Imperial Beach municipal beach, other",32.566577,-117.133205
49,637,PL-010,"Imperial Beach municipal beach, other",32.584700,-117.133000
50,321,EH-023,Imperial Beach pier area,32.578155,-117.133347
51,322,EH-030,Imperial Beach pier area,32.578750,-117.133186
52,323,EH-033,Imperial Beach pier area,32.581041,-117.133452


In [ ]:
#Preparing tract-level proximity fields for later monitoring-priority

tract_points = south_bay_tracts_proj[["geoid", "tract_name", "tract_point"]].copy()

tract_points = gpd.GeoDataFrame(
    tract_points,
    geometry="tract_point",
    crs=south_bay_tracts_proj.crs
)

impact_points_small = impact_beach_points_proj[[
    "station_id",
    "station_name",
    "beach_name",
    "geometry"
]].copy()

nearest_impact = gpd.sjoin_nearest(
    tract_points,
    impact_points_small,
    how="left",
    distance_col="nearest_impact_station_m"
)

nearest_impact = nearest_impact.rename(columns={
    "station_id": "nearest_impact_station_id",
    "station_name": "nearest_impact_station_name",
    "beach_name": "nearest_impact_beach_name"
})

nearest_impact["nearest_impact_station_km"] = nearest_impact["nearest_impact_station_m"] / 1000

nearest_impact = nearest_impact[[
    "geoid",
    "nearest_impact_station_id",
    "nearest_impact_station_name",
    "nearest_impact_beach_name",
    "nearest_impact_station_m",
    "nearest_impact_station_km"
]]

south_bay_proximity_proj = south_bay_proximity_proj.merge(
    nearest_impact,
    on="geoid",
    how="left"
)

check = pd.DataFrame({
    "rows": [len(south_bay_proximity_proj)],
    "missing_nearest_impact_station": [south_bay_proximity_proj["nearest_impact_station_id"].isna().sum()],
    "min_distance_km": [south_bay_proximity_proj["nearest_impact_station_km"].min()],
    "max_distance_km": [south_bay_proximity_proj["nearest_impact_station_km"].max()]
})

print("Processing summary.")
display(check)

display(south_bay_proximity_proj[[
    "geoid",
    "tract_name",
    "nearest_impact_beach_name",
    "nearest_impact_station_name",
    "nearest_impact_station_km"
]].head())

Processing summary.


,rows,missing_nearest_impact_station,min_distance_km,max_distance_km
0,263,0,0.277714,43.560978


,geoid,tract_name,nearest_impact_beach_name,nearest_impact_station_name,nearest_impact_station_km
0,06073010015,Census Tract 100.15,Tijana River,TJ-010,9.408292
1,06073021900,Census Tract 219,north Imperial Beach,EH-041,8.507045
2,06073013421,Census Tract 134.21,Tijana River,TJ-020,10.988857
3,06073000400,Census Tract 4,north Imperial Beach,EH-041,18.618967
4,06073000600,Census Tract 6,north Imperial Beach,EH-041,18.417445


In [ ]:
# Determines whether H2S station proximity can be added.
# Creates H2S station points only from available coordinate fields.

h2s_station_coords = h2s_rtma[[
    "location_clean",
    "target_lat",
    "target_lon"
]].drop_duplicates().copy()

h2s_coord_check = h2s_station_coords.groupby("location_clean").agg(
    rows=("location_clean", "size"),
    unique_lat=("target_lat", "nunique"),
    unique_lon=("target_lon", "nunique")
).reset_index()

h2s_station_points = h2s_station_coords.groupby("location_clean", as_index=False).agg({
    "target_lat": "first",
    "target_lon": "first"
})

h2s_station_points = gpd.GeoDataFrame(
    h2s_station_points,
    geometry=gpd.points_from_xy(
        h2s_station_points["target_lon"],
        h2s_station_points["target_lat"]
    ),
    crs="EPSG:4326"
)

h2s_station_points_proj = h2s_station_points.to_crs(target_crs)

print("Processing summary.")
display(h2s_coord_check)

print("\nH2S station points:")
display(h2s_station_points[[
    "location_clean",
    "target_lat",
    "target_lon"
]])

Processing summary.


,location_clean,rows,unique_lat,unique_lon
0,IB CIVIC CTR,1,1,1
1,NESTOR - BES,1,1,1
2,SAN YSIDRO,1,1,1



H2S station points:


,location_clean,target_lat,target_lon
0,IB CIVIC CTR,32.5846,-117.1131
1,NESTOR - BES,32.5790,-117.0910
2,SAN YSIDRO,32.5520,-117.0431


In [ ]:
# Since each H2S station has one clean coordinate pair, we can safely create
# the tract-level distance to the nearest H2S station.
tract_points = south_bay_tracts_proj[["geoid", "tract_name", "tract_point"]].copy()

tract_points = gpd.GeoDataFrame(
    tract_points,
    geometry="tract_point",
    crs=south_bay_tracts_proj.crs
)

h2s_points_small = h2s_station_points_proj[[
    "location_clean",
    "geometry"
]].copy()

nearest_h2s = gpd.sjoin_nearest(
    tract_points,
    h2s_points_small,
    how="left",
    distance_col="nearest_h2s_station_m"
)

nearest_h2s = nearest_h2s.rename(columns={
    "location_clean": "nearest_h2s_station"
})

nearest_h2s["nearest_h2s_station_km"] = nearest_h2s["nearest_h2s_station_m"] / 1000

nearest_h2s = nearest_h2s[[
    "geoid",
    "nearest_h2s_station",
    "nearest_h2s_station_m",
    "nearest_h2s_station_km"
]]

south_bay_proximity_proj = south_bay_proximity_proj.merge(
    nearest_h2s,
    on="geoid",
    how="left"
)

check = pd.DataFrame({
    "rows": [len(south_bay_proximity_proj)],
    "missing_nearest_h2s_station": [south_bay_proximity_proj["nearest_h2s_station"].isna().sum()],
    "min_distance_km": [south_bay_proximity_proj["nearest_h2s_station_km"].min()],
    "max_distance_km": [south_bay_proximity_proj["nearest_h2s_station_km"].max()]
})

print("Processing summary.")
display(check)

display(south_bay_proximity_proj[[
    "geoid",
    "tract_name",
    "nearest_h2s_station",
    "nearest_h2s_station_km"
]].head())

Processing summary.


,rows,missing_nearest_h2s_station,min_distance_km,max_distance_km
0,263,0,0.221846,44.569544


,geoid,tract_name,nearest_h2s_station,nearest_h2s_station_km
0,06073010015,Census Tract 100.15,SAN YSIDRO,9.038659
1,06073021900,Census Tract 219,IB CIVIC CTR,8.715545
2,06073013421,Census Tract 134.21,NESTOR - BES,10.049048
3,06073000400,Census Tract 4,IB CIVIC CTR,19.312153
4,06073000600,Census Tract 6,IB CIVIC CTR,19.018232


In [ ]:
# now we finalizes census tract proximity fields for later mapping and analysis.

south_bay_output = south_bay_proximity_proj.copy()

text_cols = [
    "nearest_beach_name",
    "nearest_impact_beach_name"
]

for col in text_cols:
    south_bay_output[col] = south_bay_output[col].replace("Tijana River", "Tijuana River")

distance_cols = [
    "nearest_beach_station_m",
    "nearest_beach_station_km",
    "nearest_impact_station_m",
    "nearest_impact_station_km",
    "nearest_h2s_station_m",
    "nearest_h2s_station_km"
]

for col in distance_cols:
    south_bay_output[col] = south_bay_output[col].round(3)

tract_points_wgs84 = south_bay_output.set_geometry("tract_point").to_crs("EPSG:4326")

south_bay_output["tract_point_lon"] = tract_points_wgs84.geometry.x
south_bay_output["tract_point_lat"] = tract_points_wgs84.geometry.y

south_bay_output = south_bay_output.drop(columns=["tract_point"])
south_bay_output = south_bay_output.to_crs("EPSG:4326")

output_check = pd.DataFrame({
    "rows": [len(south_bay_output)],
    "columns": [south_bay_output.shape[1]],
    "crs": [str(south_bay_output.crs)],
    "missing_geometry": [south_bay_output.geometry.isna().sum()],
    "missing_nearest_beach": [south_bay_output["nearest_beach_station_id"].isna().sum()],
    "missing_nearest_impact": [south_bay_output["nearest_impact_station_id"].isna().sum()],
    "missing_nearest_h2s": [south_bay_output["nearest_h2s_station"].isna().sum()]
})

print("Output verification complete.")
display(output_check)

display(south_bay_output[[
    "geoid",
    "tract_name",
    "tract_point_lat",
    "tract_point_lon",
    "nearest_beach_station_km",
    "nearest_impact_station_km",
    "nearest_h2s_station_km"
]].head())

Output verification complete.


,rows,columns,crs,missing_geometry,missing_nearest_beach,missing_nearest_impact,missing_nearest_h2s
0,263,29,EPSG:4326,0,0,0,0


,geoid,tract_name,tract_point_lat,tract_point_lon,nearest_beach_station_km,nearest_impact_station_km,nearest_h2s_station_km
0,06073010015,Census Tract 100.15,32.561937,-116.947732,9.408,9.408,9.039
1,06073021900,Census Tract 219,32.663265,-117.116330,1.005,8.507,8.716
2,06073013421,Census Tract 134.21,32.644948,-117.017560,6.922,10.989,10.049
3,06073000400,Census Tract 4,32.753744,-117.163200,3.995,18.619,19.312
4,06073000600,Census Tract 6,32.753123,-117.152151,4.351,18.417,19.018


In [ ]:
# save ensus tract proximity layer for analysis

south_bay_output.to_file(output_geojson, driver="GeoJSON")

south_bay_attributes = south_bay_output.drop(columns="geometry")
south_bay_attributes.to_csv(output_csv, index=False)

summary_rows = [
    {"item": "input_sd_tracts_rows", "value": len(sd_tracts)},
    {"item": "input_south_bay_tracts_rows", "value": len(south_bay_tracts)},
    {"item": "input_beach_station_rows", "value": len(beach_stations)},
    {"item": "impact_area_station_count", "value": len(impact_beach_points_proj)},
    {"item": "h2s_station_count", "value": len(h2s_station_points_proj)},
    {"item": "output_rows", "value": len(south_bay_output)},
    {"item": "output_columns", "value": south_bay_output.shape[1]},
    {"item": "output_crs", "value": str(south_bay_output.crs)},
    {"item": "missing_geometry", "value": int(south_bay_output.geometry.isna().sum())},
    {"item": "missing_nearest_beach_station", "value": int(south_bay_output["nearest_beach_station_id"].isna().sum())},
    {"item": "missing_nearest_impact_station", "value": int(south_bay_output["nearest_impact_station_id"].isna().sum())},
    {"item": "missing_nearest_h2s_station", "value": int(south_bay_output["nearest_h2s_station"].isna().sum())},
    {"item": "output_geojson_exists", "value": output_geojson.exists()},
    {"item": "output_csv_exists", "value": output_csv.exists()}
]

processing_summary = pd.DataFrame(summary_rows)
processing_summary.to_csv(summary_file, index=False)

print("File check complete.")
print(output_geojson)
print(output_csv)
print(summary_file)

display(processing_summary)

File check complete.
/content/drive/MyDrive/STAT596/Project596_datafiles/processed_data/south_bay_census_tracts_proximity.geojson
/content/drive/MyDrive/STAT596/Project596_datafiles/processed_data/south_bay_census_tracts_proximity_attributes.csv
/content/drive/MyDrive/STAT596/Project596_datafiles/tables/census_spatial_layers_processing_summary.csv


,item,value
0,input_sd_tracts_rows,737
1,input_south_bay_tracts_rows,263
2,input_beach_station_rows,284
3,impact_area_station_count,18
4,h2s_station_count,3
5,output_rows,263
6,output_columns,29
7,output_crs,EPSG:4326
8,missing_geometry,0
9,missing_nearest_beach_station,0


In [ ]:
# as usual lets check the work
saved_geojson = gpd.read_file(output_geojson)
saved_csv = pd.read_csv(output_csv)
saved_summary = pd.read_csv(summary_file)

key_fields = [
    "geoid",
    "tract_name",
    "nearest_beach_station_km",
    "nearest_impact_station_km",
    "nearest_h2s_station_km",
    "tract_point_lat",
    "tract_point_lon"
]

qa_check = pd.DataFrame({
    "file": [
        output_geojson.name,
        output_csv.name,
        summary_file.name
    ],
    "exists": [
        output_geojson.exists(),
        output_csv.exists(),
        summary_file.exists()
    ],
    "rows": [
        len(saved_geojson),
        len(saved_csv),
        len(saved_summary)
    ],
    "columns": [
        saved_geojson.shape[1],
        saved_csv.shape[1],
        saved_summary.shape[1]
    ]
})

missing_key_fields = saved_csv[key_fields].isna().sum().reset_index()
missing_key_fields.columns = ["field", "missing_values"]


print("\nSaved file check:")
display(qa_check)

print("\nGeoJSON CRS:")
print(saved_geojson.crs)

print("\nMissing values in key output fields:")
display(missing_key_fields)

print("\nSaved attribute preview:")
display(saved_csv[key_fields].head())

Final QA complete.

Saved file check:


,file,exists,rows,columns
0,south_bay_census_tracts_proximity.geojson,True,263,29
1,south_bay_census_tracts_proximity_attributes.csv,True,263,28
2,census_spatial_layers_processing_summary.csv,True,14,2



GeoJSON CRS:
EPSG:4326

Missing values in key output fields:


,field,missing_values
0,geoid,0
1,tract_name,0
2,nearest_beach_station_km,0
3,nearest_impact_station_km,0
4,nearest_h2s_station_km,0
5,tract_point_lat,0
6,tract_point_lon,0



Saved attribute preview:


,geoid,tract_name,nearest_beach_station_km,nearest_impact_station_km,nearest_h2s_station_km,tract_point_lat,tract_point_lon
0,6073010015,Census Tract 100.15,9.408,9.408,9.039,32.561937,-116.947732
1,6073021900,Census Tract 219,1.005,8.507,8.716,32.663265,-117.116330
2,6073013421,Census Tract 134.21,6.922,10.989,10.049,32.644948,-117.017560
3,6073000400,Census Tract 4,3.995,18.619,19.312,32.753744,-117.163200
4,6073000600,Census Tract 6,4.351,18.417,19.018,32.753123,-117.152151


In [ ]:
# after looking at the output. you can see geoid has no leading zero
# probably becuase I used pd.read_csv which is known to guess if something is a
# number . I rather fix here than deal with it later
# lets make sure census geoid values are 11-character text ids

saved_csv = pd.read_csv(output_csv, dtype={"geoid": str})

saved_csv["geoid"] = saved_csv["geoid"].str.zfill(11)

saved_csv.to_csv(output_csv, index=False)

saved_csv_check = pd.read_csv(output_csv, dtype={"geoid": str})

geoid_length_check = saved_csv_check["geoid"].str.len().value_counts().reset_index()
geoid_length_check.columns = ["geoid_length", "count"]

print("GEOID check complete.")

print("\nGEOID length check:")
display(geoid_length_check)

print("\nSaved CSV preview:")
display(saved_csv_check[[
    "geoid",
    "tract_name",
    "nearest_beach_station_km",
    "nearest_impact_station_km",
    "nearest_h2s_station_km"
]].head())

GEOID check complete.

GEOID length check:


,geoid_length,count
0,11,263



Saved CSV preview:


,geoid,tract_name,nearest_beach_station_km,nearest_impact_station_km,nearest_h2s_station_km
0,06073010015,Census Tract 100.15,9.408,9.408,9.039
1,06073021900,Census Tract 219,1.005,8.507,8.716
2,06073013421,Census Tract 134.21,6.922,10.989,10.049
3,06073000400,Census Tract 4,3.995,18.619,19.312
4,06073000600,Census Tract 6,4.351,18.417,19.018


In [ ]:
# now we have most datasets ready lets do an inventory to make sure
# everything is consistant,saved and readable

from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np

from google.colab import drive
drive.mount("/content/drive")

DATA_RAW = Path("/content/drive/MyDrive/STAT596/Project596_datafiles")
DATA_PROCESSED = DATA_RAW / "processed_data"

processed_files = {
    "rainfall": DATA_PROCESSED / "prism_daily_rainfall_south_bay.csv",
    "daily_beach_activity_long": DATA_PROCESSED / "daily_beach_activity_long.csv",
    "daily_beach_closure_activity": DATA_PROCESSED / "daily_beach_closure_activity.csv",
    "river_flow": DATA_PROCESSED / "tijuana_river_daily_flow_2023_2026.csv",
    "h2s_events": DATA_PROCESSED / "h2s_events_clean_2024_2025.csv",
    "h2s_daily_station_summary": DATA_PROCESSED / "h2s_daily_station_summary_2024_2025.csv",
    "h2s_rtma": DATA_PROCESSED / "h2s_rtma_matched_events_clean.csv",
    "beach_stations": DATA_PROCESSED / "beach_monitoring_stations_clean.csv",
    "census_tracts_proximity_geojson": DATA_PROCESSED / "south_bay_census_tracts_proximity.geojson",
    "census_tracts_proximity_csv": DATA_PROCESSED / "south_bay_census_tracts_proximity_attributes.csv",
}

file_check = pd.DataFrame({
    "dataset": list(processed_files.keys()),
    "path": [str(path) for path in processed_files.values()],
    "exists": [path.exists() for path in processed_files.values()]
})

print("Input check complete.")
display(file_check)

missing_files = file_check.loc[~file_check["exists"], "dataset"].tolist()

if missing_files:
    raise FileNotFoundError(f"Missing processed files: {missing_files}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Input check complete.


,dataset,path,exists
0,rainfall,/content/drive/MyDrive/STAT596/Project596_data...,True
1,daily_beach_activity_long,/content/drive/MyDrive/STAT596/Project596_data...,True
2,daily_beach_closure_activity,/content/drive/MyDrive/STAT596/Project596_data...,True
3,river_flow,/content/drive/MyDrive/STAT596/Project596_data...,True
4,h2s_events,/content/drive/MyDrive/STAT596/Project596_data...,True
5,h2s_daily_station_summary,/content/drive/MyDrive/STAT596/Project596_data...,True
6,h2s_rtma,/content/drive/MyDrive/STAT596/Project596_data...,True
7,beach_stations,/content/drive/MyDrive/STAT596/Project596_data...,True
8,census_tracts_proximity_geojson,/content/drive/MyDrive/STAT596/Project596_data...,True
9,census_tracts_proximity_csv,/content/drive/MyDrive/STAT596/Project596_data...,True


In [ ]:
# still Checking that processed datasets are readable and consistent.

rainfall = pd.read_csv(processed_files["rainfall"], parse_dates=["date"])
daily_beach_long = pd.read_csv(processed_files["daily_beach_activity_long"], parse_dates=["date"])
daily_beach = pd.read_csv(processed_files["daily_beach_closure_activity"], parse_dates=["date"])
river_flow = pd.read_csv(processed_files["river_flow"], parse_dates=["date"])
h2s_events = pd.read_csv(processed_files["h2s_events"])
h2s_daily = pd.read_csv(processed_files["h2s_daily_station_summary"])
h2s_rtma = pd.read_csv(processed_files["h2s_rtma"])
beach_stations = pd.read_csv(processed_files["beach_stations"])
census_geo = gpd.read_file(processed_files["census_tracts_proximity_geojson"])
census_csv = pd.read_csv(processed_files["census_tracts_proximity_csv"], dtype={"geoid": str})

datasets = {
    "rainfall": rainfall,
    "daily_beach_activity_long": daily_beach_long,
    "daily_beach_closure_activity": daily_beach,
    "river_flow": river_flow,
    "h2s_events": h2s_events,
    "h2s_daily_station_summary": h2s_daily,
    "h2s_rtma": h2s_rtma,
    "beach_stations": beach_stations,
    "census_tracts_proximity_geojson": census_geo,
    "census_tracts_proximity_csv": census_csv,
}

checks = []

for name, df in datasets.items():
    date_col = "date" if "date" in df.columns else None

    checks.append({
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1],
        "date_min": df[date_col].min() if date_col else np.nan,
        "date_max": df[date_col].max() if date_col else np.nan,
        "unique_dates": df[date_col].nunique() if date_col else np.nan,
        "crs": str(df.crs) if hasattr(df, "crs") else np.nan
    })

data_check = pd.DataFrame(checks)

print("Data check complete.")
display(data_check)

Data check complete.


,dataset,rows,columns,date_min,date_max,unique_dates,crs
0,rainfall,1159,6,2023-01-01 00:00:00,2026-03-04 00:00:00,1159.0,NaN
1,daily_beach_activity_long,12915,36,2023-01-01 00:00:00,2026-03-04 00:00:00,1159.0,NaN
2,daily_beach_closure_activity,1159,41,2023-01-01 00:00:00,2026-03-04 00:00:00,1159.0,NaN
3,river_flow,1159,32,2023-01-01 00:00:00,2026-03-04 00:00:00,1159.0,NaN
4,h2s_events,597,25,2024-09-12,2025-08-26,26.0,NaN
5,h2s_daily_station_summary,59,20,2024-09-12,2025-08-26,26.0,NaN
6,h2s_rtma,60,61,2024-09-25,2025-07-08,11.0,NaN
7,beach_stations,284,34,NaN,NaN,NaN,NaN
8,census_tracts_proximity_geojson,263,29,NaN,NaN,NaN,EPSG:4326
9,census_tracts_proximity_csv,263,28,NaN,NaN,NaN,NaN


In [ ]:
# Checking key fields across the final processed datasets

key_checks = []

checks_to_run = {
    "rainfall": ["date", "rain_mm", "rain_lag1_mm", "rain_3day_mm", "rain_7day_mm", "rainy_day"],
    "daily_beach_closure_activity": ["date"],
    "river_flow": ["date", "discharge_m3s", "discharge_cfs", "discharge_mgd", "high_flow_day"],
    "h2s_events": ["station_name_clean", "location_clean"],
    "h2s_daily_station_summary": ["date", "location_clean"],
    "h2s_rtma": ["location_clean", "target_lat", "target_lon"],
    "beach_stations": ["station_id", "station_name", "beach_name", "station_lat", "station_lon"],
    "census_tracts_proximity_csv": ["geoid", "tract_name", "nearest_beach_station_km", "nearest_impact_station_km", "nearest_h2s_station_km"]
}

for dataset_name, fields in checks_to_run.items():
    df = datasets[dataset_name]

    for field in fields:
        key_checks.append({
            "dataset": dataset_name,
            "field": field,
            "field_exists": field in df.columns,
            "missing_values": df[field].isna().sum() if field in df.columns else np.nan
        })

key_check_df = pd.DataFrame(key_checks)

duplicate_date_check = pd.DataFrame({
    "dataset": ["rainfall", "daily_beach_closure_activity", "river_flow"],
    "duplicate_dates": [
        rainfall["date"].duplicated().sum(),
        daily_beach["date"].duplicated().sum(),
        river_flow["date"].duplicated().sum()
    ]
})

geoid_check = pd.DataFrame({
    "dataset": ["census_tracts_proximity_csv"],
    "rows": [len(census_csv)],
    "geoid_length_11": [(census_csv["geoid"].str.len() == 11).sum()],
    "geoid_length_issue": [(census_csv["geoid"].str.len() != 11).sum()],
    "duplicate_geoid": [census_csv["geoid"].duplicated().sum()]
})

print("Key field check complete.")

print("\nMissing values in key fields:")
display(key_check_df)

print("\nDuplicate date check:")
display(duplicate_date_check)

print("\nGEOID check:")
display(geoid_check)

Key field check complete.

Missing values in key fields:


,dataset,field,field_exists,missing_values
0,rainfall,date,True,0
1,rainfall,rain_mm,True,0
2,rainfall,rain_lag1_mm,True,1
3,rainfall,rain_3day_mm,True,0
4,rainfall,rain_7day_mm,True,0
5,rainfall,rainy_day,True,0
6,daily_beach_closure_activity,date,True,0
7,river_flow,date,True,0
8,river_flow,discharge_m3s,True,0
9,river_flow,discharge_cfs,True,0



Duplicate date check:


,dataset,duplicate_dates
0,rainfall,0
1,daily_beach_closure_activity,0
2,river_flow,0



GEOID check:


,dataset,rows,geoid_length_11,geoid_length_issue,duplicate_geoid
0,census_tracts_proximity_csv,263,263,0,0


In [ ]:
# Checking that daily rainfall, beach activity, and river flow align by date

rainfall["date"] = pd.to_datetime(rainfall["date"])
daily_beach["date"] = pd.to_datetime(daily_beach["date"])
river_flow["date"] = pd.to_datetime(river_flow["date"])

rain_dates = set(rainfall["date"])
beach_dates = set(daily_beach["date"])
river_dates = set(river_flow["date"])

timeline_check = pd.DataFrame({
    "check": [
        "rainfall_dates",
        "beach_daily_dates",
        "river_flow_dates",
        "rainfall_matches_beach",
        "rainfall_matches_river",
        "beach_matches_river"
    ],
    "value": [
        len(rain_dates),
        len(beach_dates),
        len(river_dates),
        rain_dates == beach_dates,
        rain_dates == river_dates,
        beach_dates == river_dates
    ]
})

missing_from_rainfall = sorted(list((beach_dates | river_dates) - rain_dates))
missing_from_beach = sorted(list((rain_dates | river_dates) - beach_dates))
missing_from_river = sorted(list((rain_dates | beach_dates) - river_dates))

print("Timeline check complete.")
display(timeline_check)

print("\nMissing from rainfall:", len(missing_from_rainfall))
print("Missing from beach daily:", len(missing_from_beach))
print("Missing from river flow:", len(missing_from_river))

Timeline check complete.


,check,value
0,rainfall_dates,1159
1,beach_daily_dates,1159
2,river_flow_dates,1159
3,rainfall_matches_beach,True
4,rainfall_matches_river,True
5,beach_matches_river,True



Missing from rainfall: 0
Missing from beach daily: 0
Missing from river flow: 0


In [ ]:
# checking H2S station names, dates, and coordinate readiness.

h2s_events["date"] = pd.to_datetime(h2s_events["date"])
h2s_daily["date"] = pd.to_datetime(h2s_daily["date"])
h2s_rtma["date"] = pd.to_datetime(h2s_rtma["date"])

h2s_location_check = h2s_events["location_clean"].value_counts().reset_index()
h2s_location_check.columns = ["location_clean", "h2s_event_rows"]

h2s_daily_location_check = h2s_daily["location_clean"].value_counts().reset_index()
h2s_daily_location_check.columns = ["location_clean", "h2s_daily_rows"]

rtma_location_check = h2s_rtma["location_clean"].value_counts().reset_index()
rtma_location_check.columns = ["location_clean", "h2s_rtma_rows"]

h2s_coordinate_check = h2s_rtma.groupby("location_clean").agg(
    unique_target_lat=("target_lat", "nunique"),
    unique_target_lon=("target_lon", "nunique"),
    missing_target_lat=("target_lat", lambda x: x.isna().sum()),
    missing_target_lon=("target_lon", lambda x: x.isna().sum())
).reset_index()

print("H2S QA complete.")

print("\nH2S event rows by location:")
display(h2s_location_check)

print("\nH2S daily rows by location:")
display(h2s_daily_location_check)

print("\nH2S RTMA rows by location:")
display(rtma_location_check)

print("\nH2S RTMA coordinate check:")
display(h2s_coordinate_check)

H2S QA complete.

H2S event rows by location:


,location_clean,h2s_event_rows
0,SAN YSIDRO,235
1,NESTOR - BES,219
2,IB CIVIC CTR,143



H2S daily rows by location:


,location_clean,h2s_daily_rows
0,SAN YSIDRO,23
1,NESTOR - BES,22
2,IB CIVIC CTR,14



H2S RTMA rows by location:


,location_clean,h2s_rtma_rows
0,NESTOR - BES,31
1,SAN YSIDRO,20
2,IB CIVIC CTR,9



H2S RTMA coordinate check:


,location_clean,unique_target_lat,unique_target_lon,missing_target_lat,missing_target_lon
0,IB CIVIC CTR,1,1,0,0
1,NESTOR - BES,1,1,0,0
2,SAN YSIDRO,1,1,0,0


In [ ]:
# checking if H2S dates align with the main daily project timeline.

main_daily_dates = set(daily_beach["date"])

h2s_events_dates = set(h2s_events["date"])
h2s_daily_dates = set(h2s_daily["date"])
h2s_rtma_dates = set(h2s_rtma["date"])

h2s_date_check = pd.DataFrame({
    "dataset": ["h2s_events", "h2s_daily_station_summary", "h2s_rtma"],
    "unique_dates": [
        len(h2s_events_dates),
        len(h2s_daily_dates),
        len(h2s_rtma_dates)
    ],
    "date_min": [
        h2s_events["date"].min(),
        h2s_daily["date"].min(),
        h2s_rtma["date"].min()
    ],
    "date_max": [
        h2s_events["date"].max(),
        h2s_daily["date"].max(),
        h2s_rtma["date"].max()
    ],
    "dates_missing_from_main_daily_timeline": [
        len(h2s_events_dates - main_daily_dates),
        len(h2s_daily_dates - main_daily_dates),
        len(h2s_rtma_dates - main_daily_dates)
    ]
})

print("H2S timeline check complete.")
display(h2s_date_check)

H2S timeline check complete.


,dataset,unique_dates,date_min,date_max,dates_missing_from_main_daily_timeline
0,h2s_events,26,2024-09-12,2025-08-26,0
1,h2s_daily_station_summary,26,2024-09-12,2025-08-26,0
2,h2s_rtma,11,2024-09-25,2025-07-08,0


In [ ]:
# one final check
readiness_check = pd.DataFrame([
    {
        "dataset": "rainfall",
        "status": "ready",
        "main_use_later": "daily rainfall, lag rainfall, rainy-day comparisons"
    },
    {
        "dataset": "daily_beach_closure_activity",
        "status": "ready",
        "main_use_later": "daily beach active record-days and date joins"
    },
    {
        "dataset": "daily_beach_activity_long",
        "status": "ready",
        "main_use_later": "beach-level and station-level summaries"
    },
    {
        "dataset": "river_flow",
        "status": "ready",
        "main_use_later": "daily flow, lag flow, high-flow windows"
    },
    {
        "dataset": "h2s_events",
        "status": "ready",
        "main_use_later": "elevated H2S event patterns by station and date"
    },
    {
        "dataset": "h2s_rtma",
        "status": "ready",
        "main_use_later": "wind and weather during stronger H2S events"
    },
    {
        "dataset": "beach_stations",
        "status": "ready",
        "main_use_later": "beach monitoring station mapping"
    },
    {
        "dataset": "census_tracts_proximity",
        "status": "ready",
        "main_use_later": "community proximity mapping and monitoring-priority interpretation"
    }
])

print("Data readiness check complete.")
display(readiness_check)

Data readiness check complete.


,dataset,status,main_use_later
0,rainfall,ready,"daily rainfall, lag rainfall, rainy-day compar..."
1,daily_beach_closure_activity,ready,daily beach active record-days and date joins
2,daily_beach_activity_long,ready,beach-level and station-level summaries
3,river_flow,ready,"daily flow, lag flow, high-flow windows"
4,h2s_events,ready,elevated H2S event patterns by station and date
5,h2s_rtma,ready,wind and weather during stronger H2S events
6,beach_stations,ready,beach monitoring station mapping
7,census_tracts_proximity,ready,community proximity mapping and monitoring-pri...
